In [1]:
print("HEllo world")

HEllo world


## Docs, References
https://reference.langchain.com/python/langchain-core/runnables/base/Runnable

In [2]:
from helpers.common import (
  BASE_URL,
  DEEPSEEK_MODEL,
  GEMMA_MODEL,
  langfuse,
  langfuse_handler,
)



In [3]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser



In [10]:
llm = ChatOllama(
  model=GEMMA_MODEL,
  base_url=BASE_URL,
  validate_model_on_init=True
)


In [11]:
system_prompt = SystemMessagePromptTemplate.from_template("You are a {level} teacher. Answer in short points.")
human_prompt = HumanMessagePromptTemplate.from_template("Tell me about {topic} in {number} points.")
chat_prompt = ChatPromptTemplate([system_prompt, human_prompt])
chat_prompt

ChatPromptTemplate(input_variables=['level', 'number', 'topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['level'], input_types={}, partial_variables={}, template='You are a {level} teacher. Answer in short points.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['number', 'topic'], input_types={}, partial_variables={}, template='Tell me about {topic} in {number} points.'), additional_kwargs={})])

## Without Chain

In [25]:
# final_chat_prompt = chat_prompt.invoke({
#   'language': "hindi"
# })

# final_chat_prompt
# async for message in llm.astream(
#   final_chat_prompt,
#   config={"callbacks": [langfuse_handler], "run_name": "ollama-translator"},
# ):
#   print(message.content, end="")

# With Chain
### Sequential

In [12]:
chain = chat_prompt | llm
chain

ChatPromptTemplate(input_variables=['level', 'number', 'topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['level'], input_types={}, partial_variables={}, template='You are a {level} teacher. Answer in short points.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['number', 'topic'], input_types={}, partial_variables={}, template='Tell me about {topic} in {number} points.'), additional_kwargs={})])
| ChatOllama(model='gemma3:4b', validate_model_on_init=True, base_url='http://localhost:11434/')

In [ ]:
for message in chain.stream({
  "level": "school",
  "topic": "Earth",
  "number": "2",

},
  config={"callbacks": [langfuse_handler], "run_name": "school-teacher"},
):
  print(message.content, end="")


Okay class, here’s Earth in two key points:

1.  **Unique Habitable Planet:** Earth is the only planet we know of with liquid water on its surface – crucial for life as we know it!
2.  **Dynamic & Changing:** Our planet is constantly being shaped by forces like plate tectonics, weather, and erosion. 

---

Do you want me to elaborate on any of those points, or perhaps give you a different fact about Earth?

# Chain Sequential | StrOutputParser

In [13]:
chain_1 = chat_prompt | llm | StrOutputParser()
chain_1

ChatPromptTemplate(input_variables=['level', 'number', 'topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['level'], input_types={}, partial_variables={}, template='You are a {level} teacher. Answer in short points.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['number', 'topic'], input_types={}, partial_variables={}, template='Tell me about {topic} in {number} points.'), additional_kwargs={})])
| ChatOllama(model='gemma3:4b', validate_model_on_init=True, base_url='http://localhost:11434/')
| StrOutputParser()

In [29]:
# for message in chain_2.stream({
#   "level": "school",
#   "topic": "Earth",
#   "number": "2",

# },
#   config={"callbacks": [langfuse_handler], "run_name": "school-teacher"},
# ):
#   print(message, end="")
response = chain_1.invoke({
  "level": "school",
  "topic": "Earth",
  "number": "2",
})
response

'Okay class, here’s Earth in two key points:\n\n1.  **Unique Habitable Planet:** Earth is the only planet we know of with liquid water on its surface, which is essential for life as we know it.\n2.  **Dynamic & Changing:** Our planet is constantly being shaped by forces like plate tectonics, weather, and erosion – it’s a very active place! \n\n---\n\nDo you want me to elaborate on any of those points, or perhaps ask you a question about it?'

##  Chain Multiple Runnalbes
### One after the other

###### Example: You get the output from ek LLM. Woh output you send it to the LLM again to evaluate

In [14]:
## Analysis Code
analysis_prompt = ChatPromptTemplate.from_template("""
Analyze the following text: {response}. You need to tell how how difficult it is to understand. Answer in one sentence only.
""")


chain_2 = analysis_prompt | llm | StrOutputParser()
# chain_2.invoke({
#   'response': response
# },
#   config={
#   "callbacks": [langfuse_handler], 
#   "run_name": "analysis-chain",
#     "metadata": {"langfuse_tags": ["open-ai"]},
#   },
# )


## Combining Chains

In [15]:
# composed_chain = {'response': chain_1} | analysis_prompt | llm | StrOutputParser()
# output = composed_chain.invoke({
#   "level": "school",
#   "topic": "Earth",
#   "number": "2",
# },
#   config={"callbacks": [langfuse_handler],"run_name": "composed-chain (Topic+Analysis)",    "metadata": {"langfuse_tags": ["gemma"]},},

# )
composed_chain = {'response': chain_1} | analysis_prompt | llm | StrOutputParser()

for message in composed_chain.stream({
  'level': 'phd',
  "topic": "Earth",
  "number": "2",
},
config={"callbacks": [langfuse_handler],"run_name": "composed-chain (Topic+Analysis)",    "metadata": {"langfuse_tags": ["gemma"]},},
):
  print(message, end="")


This text is relatively easy to understand, presenting core concepts about Earth's nature in a clear and concise manner.

## Parallel LCEL

In [25]:
## poem chain
human_message = HumanMessagePromptTemplate.from_template("Write a poem on {topic} in {number} lines.")
chat_prompt = ChatPromptTemplate([human_message])


poem_chain = chat_prompt | llm | StrOutputParser()
poem_chain

ChatPromptTemplate(input_variables=['number', 'topic'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['number', 'topic'], input_types={}, partial_variables={}, template='Write a poem on {topic} in {number} lines.'), additional_kwargs={})])
| ChatOllama(model='gemma3:4b', validate_model_on_init=True, base_url='http://localhost:11434/')
| StrOutputParser()

In [26]:
for message in poem_chain.invoke({
  "topic": "Food",
  "number": 3
},
config={"callbacks": [langfuse_handler],"run_name": "Poem Chain",    "metadata": {"langfuse_tags": ["poem"]},},
):
  print(message, end="")


A vibrant burst, a fragrant plea,
Nourishing bodies, delightfully free,
Taste and comfort, for you and me.

In [31]:
# fact chain
fact_system_message = SystemMessagePromptTemplate.from_template("You are a {level} teacher. Answer in short sentences.")
fact_human_message = HumanMessagePromptTemplate.from_template("Tell me about {topic} in {number} of lines.")
messages = [fact_system_message, fact_human_message]
fact_chat_prompt =  ChatPromptTemplate(messages)

fact_chain = fact_chat_prompt | llm | StrOutputParser()
fact_chain

ChatPromptTemplate(input_variables=['level', 'number', 'topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['level'], input_types={}, partial_variables={}, template='You are a {level} teacher. Answer in short sentences.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['number', 'topic'], input_types={}, partial_variables={}, template='Tell me about {topic} in {number} of lines.'), additional_kwargs={})])
| ChatOllama(model='gemma3:4b', validate_model_on_init=True, base_url='http://localhost:11434/')
| StrOutputParser()

In [32]:
for message in fact_chain.invoke({
  "level": "school",
  "topic": "Food",
  "number": 3
},
config={"callbacks": [langfuse_handler],"run_name": "Food Chain",    "metadata": {"langfuse_tags": ["poem"]},},
):
  print(message, end="")


Okay class, let’s talk about food! 

Food gives us energy to grow and learn. 

It also comes in many different flavors and cultures. 

Do you have any questions?

## Runnable Parallel

In [35]:
from langchain_core.runnables import RunnableParallel

parallel_chain = RunnableParallel(
  poem = poem_chain,
  fact = fact_chain
)
output = parallel_chain.invoke({
  "level":"PHD",
  "topic": "Earth",
  "number": 3 
},
config={"callbacks": [langfuse_handler],"run_name": "Parallel Chain",    "metadata": {"langfuse_tags": ["parallel-chain"]},},
)


In [16]:
langfuse.flush()